In [1]:
import pandas as pd
import numpy as np

In [2]:
forecast_results = pd.read_csv(
    "../data/processed/forecast_results.csv",
    parse_dates=["date"]
)

forecast_results.head()

,date,item,actual_demand,q10,q50,q90
0,2017-07-09,1,2227,2221.549515,2277.337242,2391.773561
1,2017-07-16,1,2261,2107.127532,2248.020940,2320.767430
2,2017-07-23,1,2239,2113.073516,2284.675439,2361.518382
3,2017-07-30,1,2247,2204.192185,2281.343412,2349.492661
4,2017-08-06,1,1947,1921.010004,2112.598400,2178.813692


In [3]:
print(forecast_results.shape)
print(forecast_results["item"].nunique())
print(forecast_results["date"].nunique())

(520, 6)
20
26


In [4]:
%pip install pulp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pulp

print(pulp.__version__)

4.0.0


In [6]:
bom = {
    1:  {"RM1": 0.5, "RM2": 0.8, "RM3": 0.2},
    2:  {"RM1": 0.6, "RM2": 0.7, "RM3": 0.3},
    3:  {"RM1": 0.4, "RM2": 0.9, "RM3": 0.2},
    4:  {"RM1": 0.7, "RM2": 0.6, "RM3": 0.4},
    5:  {"RM1": 0.5, "RM2": 0.8, "RM3": 0.3},
    6:  {"RM1": 0.6, "RM2": 0.7, "RM3": 0.2},
    7:  {"RM1": 0.4, "RM2": 0.8, "RM3": 0.3},
    8:  {"RM1": 0.7, "RM2": 0.9, "RM3": 0.2},
    9:  {"RM1": 0.5, "RM2": 0.6, "RM3": 0.4},
    10: {"RM1": 0.6, "RM2": 0.8, "RM3": 0.3},
    11: {"RM1": 0.4, "RM2": 0.7, "RM3": 0.2},
    12: {"RM1": 0.7, "RM2": 0.8, "RM3": 0.3},
    13: {"RM1": 0.5, "RM2": 0.9, "RM3": 0.2},
    14: {"RM1": 0.6, "RM2": 0.6, "RM3": 0.4},
    15: {"RM1": 0.4, "RM2": 0.8, "RM3": 0.3},
    16: {"RM1": 0.7, "RM2": 0.7, "RM3": 0.2},
    17: {"RM1": 0.5, "RM2": 0.8, "RM3": 0.4},
    18: {"RM1": 0.6, "RM2": 0.9, "RM3": 0.3},
    19: {"RM1": 0.4, "RM2": 0.6, "RM3": 0.2},
    20: {"RM1": 0.7, "RM2": 0.8, "RM3": 0.3},
}

In [7]:
production_capacity = 90000

In [8]:
raw_material_capacity = {
    "RM1": 50000,
    "RM2": 75000,
    "RM3": 30000
}

In [9]:
production_cost = {
    1: 10, 2: 11, 3: 9, 4: 12, 5: 10,
    6: 11, 7: 9, 8: 12, 9: 10, 10: 11,
    11: 9, 12: 12, 13: 10, 14: 11, 15: 9,
    16: 12, 17: 10, 18: 11, 19: 9, 20: 12
}

In [10]:
holding_cost = 2

In [11]:
backlog_penalty = 25

In [12]:
print("Shape:", forecast_results.shape)
print("Items:", forecast_results["item"].nunique())
print("Weeks:", forecast_results["date"].nunique())
print("Date range:", forecast_results["date"].min(), "to", forecast_results["date"].max())

Shape: (520, 6)
Items: 20
Weeks: 26
Date range: 2017-07-09 00:00:00 to 2017-12-31 00:00:00


In [13]:
items = sorted(forecast_results["item"].unique())
weeks = sorted(forecast_results["date"].unique())

demand = (
    forecast_results
    .pivot(index="date", columns="item", values="q50")
    .loc[weeks, items]
)

print("Items:", len(items))
print("Weeks:", len(weeks))
print("Demand shape:", demand.shape)

demand.head()

Items: 20
Weeks: 26
Demand shape: (26, 20)


item,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
date,,,,,,,,,,,,,,,,,,,,
2017-07-09,2277.337242,5908.763850,3664.159003,2222.600935,1895.791759,6069.269456,5853.265990,7784.473866,5224.920010,7494.507865,7103.766450,7004.979561,8256.975944,5835.735115,8249.464911,2603.118537,3269.550676,8266.350392,4115.158668,4836.421833
2017-07-16,2248.020940,5869.554673,3773.452019,2212.532315,1884.253714,5974.744274,5929.894124,7881.332273,5199.911243,7441.744939,7176.341795,7149.452452,8248.633615,5940.636755,8234.010755,2665.090450,3330.869992,8239.412112,4152.434861,4928.765375
2017-07-23,2284.675439,5888.871055,3772.093686,2266.887735,1880.189177,5988.785143,5880.144053,7893.031074,5252.366269,7408.180083,7069.300042,7099.312207,8240.569064,5937.422796,8249.143764,2678.044495,3342.505607,8249.313978,3982.341132,4909.727193
2017-07-30,2281.343412,5910.523628,3713.170068,2161.309824,1896.350357,6124.000120,6080.599968,7983.258770,5143.796205,7491.209842,7251.977090,7351.015057,8259.320160,6032.162276,8253.207171,2609.036269,3385.695711,8284.315804,3999.630102,4937.225893
2017-08-06,2112.598400,5603.027998,3577.233895,2112.786279,1731.537575,5549.592617,5648.544452,7261.353013,4893.644507,6766.535684,6641.256697,6531.060761,8078.316131,5642.425310,8164.634145,2440.152092,3147.102599,7880.300899,3772.479889,4602.457335


In [14]:
model = pulp.LpProblem(
    "Demand_to_Production_Planner",
    pulp.LpMinimize
)

print("Optimization model created!")

Optimization model created!


In [15]:
import pulp

print(pulp.__version__)
print(pulp.LpVariable)

4.0.0
<class 'pulp.core.lp_variable.LpVariable'>


In [16]:
import inspect

print(inspect.signature(pulp.LpVariable))

(_var: '_rustcore.Variable') -> 'None'


In [17]:
production = {}

for item in items:
    for week in weeks:
        production[(item, week)] = model.add_variable(
            f"Production_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

In [18]:
inventory = {}

for item in items:
    for week in weeks:
        inventory[(item, week)] = model.add_variable(
            f"Inventory_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

In [19]:
backlog = {}

for item in items:
    for week in weeks:
        backlog[(item, week)] = model.add_variable(
            f"Backlog_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

In [20]:
print(production[(1, weeks[0])])
print(inventory[(1, weeks[0])])
print(backlog[(1, weeks[0])])

Production_1_2017-07-09_00:00:00
Inventory_1_2017-07-09_00:00:00
Backlog_1_2017-07-09_00:00:00


In [21]:
initial_inventory = {
    item: 0
    for item in items
}

initial_backlog = {
    item: 0
    for item in items
}

print(initial_inventory)
print(initial_backlog)

{np.int64(1): 0, np.int64(2): 0, np.int64(3): 0, np.int64(4): 0, np.int64(5): 0, np.int64(6): 0, np.int64(7): 0, np.int64(8): 0, np.int64(9): 0, np.int64(10): 0, np.int64(11): 0, np.int64(12): 0, np.int64(13): 0, np.int64(14): 0, np.int64(15): 0, np.int64(16): 0, np.int64(17): 0, np.int64(18): 0, np.int64(19): 0, np.int64(20): 0}
{np.int64(1): 0, np.int64(2): 0, np.int64(3): 0, np.int64(4): 0, np.int64(5): 0, np.int64(6): 0, np.int64(7): 0, np.int64(8): 0, np.int64(9): 0, np.int64(10): 0, np.int64(11): 0, np.int64(12): 0, np.int64(13): 0, np.int64(14): 0, np.int64(15): 0, np.int64(16): 0, np.int64(17): 0, np.int64(18): 0, np.int64(19): 0, np.int64(20): 0}


In [22]:
for item in items:
    for i, week in enumerate(weeks):

        if i == 0:
            previous_inventory = initial_inventory[item]
            previous_backlog = initial_backlog[item]
        else:
            previous_week = weeks[i - 1]
            previous_inventory = inventory[(item, previous_week)]
            previous_backlog = backlog[(item, previous_week)]

        model += (
            inventory[(item, week)]
            - backlog[(item, week)]
            ==
            previous_inventory
            - previous_backlog
            + production[(item, week)]
            - demand.loc[week, item]
        )

In [23]:
#he factory can produce at most 90,000 units across all 20 products in any week. - Factory production capacity
for week in weeks:
    model += (
        pulp.lpSum(
            production[(item, week)]
            for item in items
        )
        <= production_capacity
    )

print("Production capacity constraints added!")

Production capacity constraints added!


In [24]:
for week in weeks:
    for material in ["RM1", "RM2", "RM3"]:
        
        model += (
            pulp.lpSum(
                production[(item, week)] * bom[item][material]
                for item in items
            )
            <= raw_material_capacity[material]
        )

print("Raw material constraints added!")

Raw material constraints added!


In [25]:
production_cost
holding_cost = 2
backlog_penalty = 25

In [26]:

# #It will search for the sweet spot:
# Produce enough to satisfy demand while avoiding unnecessary production, inventory, and backlog.
total_cost = (
    pulp.lpSum(
        production[(item, week)] * production_cost[item]
        for item in items
        for week in weeks
    )
    +
    pulp.lpSum(
        inventory[(item, week)] * holding_cost
        for item in items
        for week in weeks
    )
    +
    pulp.lpSum(
        backlog[(item, week)] * backlog_penalty
        for item in items
        for week in weeks
    )
)

model += total_cost

print("Objective function added!")

Objective function added!


In [27]:
print(dir(pulp)[:50])

['Any', 'CHOCO_CMD', 'COINMP_DLL', 'COIN_CMD', 'COPT', 'COPT_CMD', 'COPT_DLL', 'CPLEX', 'CPLEX_CMD', 'CPLEX_PY', 'CPSAT', 'CUOPT', 'CYLP', 'EPS', 'FSCIP', 'FSCIP_CMD', 'GLPK', 'GLPK_CMD', 'GUROBI', 'GUROBI_CMD', 'HiGHS', 'HiGHS_CMD', 'Iterator', 'LpAffineExpression', 'LpBinary', 'LpCategories', 'LpConstraint', 'LpConstraintEQ', 'LpConstraintGE', 'LpConstraintLE', 'LpConstraintSenses', 'LpConstraintTypeToMps', 'LpContinuous', 'LpCplexLPLineSize', 'LpInteger', 'LpMaximize', 'LpMinimize', 'LpProblem', 'LpSenses', 'LpSensesMPS', 'LpSolveStats', 'LpSolveStatus', 'LpSolver', 'LpSolverDefault', 'LpSolver_CMD', 'LpVariable', 'MIPCL_CMD', 'MOSEK', 'PYGLPK', 'PulpError']


In [28]:
%pip install highspy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import pulp

print(pulp.listSolvers(onlyAvailable=True))

['HiGHS']


In [31]:
print("Result:", result)
print("Model status:", model.status)

Result: solver: HiGHS
status: Optimal (1)
has solution: True
objective: 75125199.9908572
time: 0.0448s (cpu 0.0469s)


AttributeError: 'LpProblem' object has no attribute 'status'

In [32]:
print("Objective:", model.objective.value())

Objective: 75125199.9908572


In [33]:
print(result.status)

1


In [34]:
optimized_plan = []

for week in weeks:
    for item in items:
        optimized_plan.append({
            "date": week,
            "item": item,
            "forecast_demand": demand.loc[week, item],
            "production": production[(item, week)].value(),
            "inventory": inventory[(item, week)].value(),
            "backlog": backlog[(item, week)].value()
        })

optimized_plan = pd.DataFrame(optimized_plan)

optimized_plan.head(20)

,date,item,forecast_demand,production,inventory,backlog
0,2017-07-09,1,2277.337242,2277.337242,0.0,0.000000
1,2017-07-09,2,5908.763850,5908.763850,0.0,0.000000
2,2017-07-09,3,3664.159003,0.000000,0.0,3664.159003
3,2017-07-09,4,2222.600935,2222.600935,0.0,0.000000
4,2017-07-09,5,1895.791759,1895.791759,0.0,0.000000
5,2017-07-09,6,6069.269456,6069.269456,0.0,0.000000
6,2017-07-09,7,5853.265990,5853.265990,0.0,0.000000
7,2017-07-09,8,7784.473866,7784.473866,0.0,0.000000
8,2017-07-09,9,5224.920010,5224.920010,0.0,0.000000
9,2017-07-09,10,7494.507865,7494.507865,0.0,0.000000


In [35]:
print("Total forecast demand:",
      optimized_plan["forecast_demand"].sum())

print("Total production:",
      optimized_plan["production"].sum())

print("Ending inventory:",
      optimized_plan["inventory"].sum())

print("Total backlog:",
      optimized_plan["backlog"].sum())

Total forecast demand: 2315264.6444241377
Total production: 2315264.6444241377
Ending inventory: 0.0
Total backlog: 2033710.42879378


In [36]:
import os

os.makedirs("../data/processed", exist_ok=True)

optimized_plan.to_csv(
    "../data/processed/optimized_plan.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
